# Churn Model Walkthrough

Reproducible notebook companion to `src/models/train_churn.py`.
Run the pipeline first (`python run_pipeline.py`) so features exist under `data/processed/`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import joblib

ROOT = Path("..") if Path("data").exists() is False else Path(".")
# Prefer project root
for cand in [Path("."), Path("..")]:
    if (cand / "data/processed/churn_features.csv").exists():
        ROOT = cand
        break

features = pd.read_csv(ROOT / "data/processed/churn_features.csv")
metrics = json.loads((ROOT / "artifacts/model/metrics.json").read_text())
model = joblib.load(ROOT / "artifacts/model/churn_model.joblib")
imp = pd.read_csv(ROOT / "artifacts/model/feature_importance.csv")

print(features.shape)
print(json.dumps({k: metrics[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]}, indent=2))
imp.head(10)


## Score a batch

Use the saved pipeline to score active customers and flag high churn probability.


In [ ]:
NUMERIC = [
    "seats", "mrr", "tenure_days", "monthly_active_days", "support_tickets_90d",
    "nps_score", "feature_adoption_score", "payment_failures_90d",
    "lifetime_revenue", "txn_count", "total_tickets", "customer_health_score",
]
CATEGORICAL = ["segment", "region", "plan", "acquisition_channel"]

active = features[features["is_churned"] == 0].copy()
active["churn_proba"] = model.predict_proba(active[NUMERIC + CATEGORICAL])[:, 1]
active.nlargest(10, "churn_proba")[["customer_id", "segment", "plan", "mrr", "customer_health_score", "churn_proba"]]
